# 🔬 Notebook 5b: Ablation 2 — Remove Feature Alignment MSE (w/o $L_{\text{feature}}$)
This notebook runs **Ablation 2**: Knowledge Distillation on Crack500 with **Intermediate Feature Alignment MSE disabled** ($eta = 0$).
* **Goal**: Measure how much representation transfer relies on intermediate backbone feature MSE.
* **Input Dataset**: Crack500 + pre-computed SAM 2 teacher logits.


In [ ]:
!mkdir -p configs utils distillation scripts checkpoints data/datasets data/teacher_logits_box runs


In [ ]:
!pip install -q ultralytics albumentations pycocotools thop pyyaml pandas


In [ ]:
import os, shutil
from pathlib import Path
input_dir = Path("/kaggle/input/distill_datasetforme")
if not input_dir.exists(): input_dir = Path("/kaggle/input")
datasets_dir = Path("data/datasets")
datasets_dir.mkdir(parents=True, exist_ok=True)

for root, dirs, files in os.walk(str(input_dir)):
    root_path = Path(root)
    if "traincrop" in dirs:
        dest = datasets_dir / "crack500"
        if os.path.lexists(dest): os.unlink(dest) if os.path.islink(dest) else shutil.rmtree(dest)
        os.symlink(root_path, dest)
        print(f"Linked Crack500: {root_path} -> {dest}")
        break


In [ ]:
# Run Ablation 2 (No Feature MSE)
import sys
sys.path.insert(0, ".")
from distillation.kd_trainer import KDSegmentationTrainer
from utils.config_loader import load_config, override_config

cfg = load_config("configs/config.yaml")
cfg = override_config(cfg, {
    "project.name": "crack_distill",
    "project.experiment": "ablation_no_feature",
    "distillation.enabled": True,
    "distillation.losses.mask_kd.enabled": True,
    "distillation.losses.feature.enabled": False,
    "distillation.losses.boundary.enabled": True,
    "teacher.logits_dir": "data/teacher_logits_box/"
})

trainer = KDSegmentationTrainer(cfg)
trainer.train()
print("✓ Ablation 2 (No Feature MSE) completed!")
